# 5. Multi Schema用法

## 5.1. 状态类型
LangGraph 支持在一个图中使用多个状态 Schema，用于区分图的外部输入、外部输出、内部共享状态以及节点间的临时状态。

常见状态类型可以分为以下几类：
- 全局状态 / 内部状态：图内部主要使用的状态，创建 StateGraph 时传递给 `state_schema` 参数。它通常包含图运行过程中需要读写的大部分字段。
- 输入状态：图对外接收输入时使用的状态，创建 StateGraph 时传递给 `input_schema` 参数。它用于约束调用图时允许传入哪些字段。
- 输出状态：图最终对外返回结果时使用的状态，创建 StateGraph 时传递给 `output_schema` 参数。它用于约束图运行结束后只返回哪些字段。
- 私有状态：图内部节点之间传递的临时状态，通常不作为图的输入，也不作为图的最终输出。它可以通过节点函数的入参类型注解声明，并在节点返回值中写入。

需要注意，输入状态和输出状态主要面向图的边界，即“图如何接收外部输入”和“图如何返回外部结果”；而全局状态和私有状态主要面向图内部节点之间的数据传递。

## 5.2 状态之间的关系

### 5.2.1 设计规范
本节主要说明 LangGraph 状态设计中的规范。以下规则属于工程上的最佳实践，违反这些规范未必一定导致程序报错，但容易降低代码的可读性和可维护性。

1. 输入状态和输出状态通常应是全局状态的子集 输入状态描述图对外需要接收的数据，输出状态描述图最终需要返回的数据。通常情况下，它们都应该是全局状态的一部分。 如：
   ```python
   class InputState(TypedDict):
       username: str

   class OutputState(TypedDict):
       graph_output: str

   class OverAllState(TypedDict):
       username: str
       nickname: str
       graph_output: str
   ```
   其中，`InputState` 和 `OutputState` 的所有字段均存在于 `OverAllState` 中。
2. 私有状态和全局状态应尽量避免字段重名 私有状态的定位是图内部某些节点之间传递的临时字段。如果私有状态字段和全局状态字段重名，虽然某些情况下程序仍然可以运行，但容易让人误以为该字段是全局共享字段，从而造成理解混乱。 因此，推荐让私有状态字段和全局状态字段保持清晰边界。
3. 节点函数应明确声明入参状态类型和返回状态类型 节点函数的第一个参数通常是当前节点可读取的状态。通过类型注解声明该参数，可以明确表达该节点需要读取哪些字段。 同时，给节点函数声明返回状态类型，也可以帮助阅读者理解该节点会更新哪些字段。 例如：
   ```python
   def node_1(state: InputState) -> OverAllState:
       return {
           "nickname": "Dear " + state["username"]
       }
   ```
4. 节点函数中不应该访问入参状态类型中不存在的字段 节点实际接收到的状态会按照其入参类型进行裁剪。因此，如果节点入参声明为 InputState，就不应该在节点内部访问 InputState 中不存在的字段。 例如：
   ```python
   def node_1(state: InputState) -> OverAllState:
       return {
           "nickname": state["username"]
       }
   ```
5. 节点函数返回的字典应尽量和返回类型注解保持一致 从 Python 类型注解的角度看，函数返回类型只是静态提示，运行时不会自动强制校验。 从 LangGraph 的运行机制看，节点返回的是对状态的部分更新，不是完整状态。只要返回字段已经被图记录为可用状态字段，LangGraph 就可以将其作为状态更新处理。 不过，从工程规范上讲，节点返回字典中的字段最好和函数返回类型注解保持一致，这样更利于阅读、调试和维护。

### 5.2.2 源码层面的约束
本节从底层机制角度说明 LangGraph 如何记录、裁剪和更新状态。

#### 5.2.2.1 状态的记录
LangGraph 的状态并不是简单保存在一个普通字典中，而是会被拆分成多个可读写的状态字段。每个状态字段在底层通常对应一个 `Channel`。

这些状态字段会在**不同阶段**被记录到状态图中。

1. `StateGraph` 记录状态字段的核心方法是 `_add_schema()`， `_add_schema() `会解析传入的状态 `Schema`，并将其中声明的字段记录到图中，使这些字段成为图运行时可以读写的状态字段。

2. 创建 `StateGraph` 时，会记录 `state_schema`、`input_schema` 和 `output_schema` 中的字段 当创建状态图时：

   ```python
   builder = StateGraph(
       OverAllState,
       input_schema=InputState,
       output_schema=OutputState
   )
   ```
    LangGraph 会解析这些 `Schema`，并将其中涉及的字段加入图的状态管理体系。

3. 调用 `add_node()` 添加节点时，也可能记录节点入参声明的状态 `Schema` 当添加节点时，LangGraph 会根据节点函数第一个参数的类型注解推断该节点的输入状态类型。 如果这个输入状态类型之前没有被图记录过，LangGraph 也会通过 `_add_schema()` 将其加入图中。 这也是私有状态能够生效的原因。 例如：

   ```python
   class PrivateState(TypedDict):
       greeting: str

   def node_3(state: PrivateState) -> OutputState:
       return {
           "graph_output": state["greeting"]
       }
   ```
    当 `node_3` 被添加到图中时，`PrivateState` 中的 `greeting` 字段会被记录到图中，从而成为图内部可以传递的状态字段。

4. 总结

- 全局状态、输入状态、输出状态通常在创建 `StateGraph` 时被记录。
- 私有状态通常在调用 `add_node()` 添加节点时，根据节点入参类型注解被记录。
- 被记录后的状态字段，底层会成为图运行时可以读写的状态字段。

#### 5.2.2.2 状态的访问
1. 调用图时，输入会按照 `input_schema` 进行约束 当调用图时：
   ```python
   graph.invoke({"username": "小黄"})
   ```
    如果创建图时声明了 `input_schema`，那么外部输入会按照 `input_schema` 进行约束。 如果没有声明 `input_schema`，则通常按照 `state_schema` 作为图的输入 `Schema`。 因此，`input_schema` 的作用不是“只让第一个节点可见”，而是约束图的外部输入结构。 此处的约束是指：**按照 `schema` 裁剪输入，只保留 `schema` 中出现的状态字段**

2. 节点接收到的状态**会按照节点入参类型进行裁剪 每个节点能读取哪些字段**，主要取决于该节点第一个参数的类型注解。 例如：

   ```python
   def node_1(state: InputState) -> OverAllState:
       ...
   ```

    此时，`node_1` 接收到的 `state` 会按照 `InputState` 进行裁剪。即使图的全局状态中还有其他字段，`node_1` 也不应该访问不属于 `InputState` 的字段。 如果访问了入参状态中不存在的字段，例如：

   ```python
   state["nickname"]
   ```
    就可能抛出：
   ```python
   KeyError
   ```
3. 节点返回的是状态更新，而不是完整状态 节点函数不需要返回完整状态，只需要返回本节点想要更新的字段。 例如：

   ```python
   def node_1(state: InputState) -> OverAllState:
       return {
           "nickname": "Dear " + state["username"]
       }
   ```
    这里虽然返回类型注解是 `OverAllState`，但函数实际只返回了 `nickname` 一个字段。这是允许的，因为 LangGraph 会把节点返回值视为对状态的部分更新。

4. 节点返回值的应用主要由字段名称和图中已记录的状态字段决定 节点返回的字典会根据字段名称写入对应状态字段，并按照该字段的 `Reducer` 规则进行合并。 需要注意的是，函数返回类型注解主要用于表达代码意图，不是严格的运行时写入边界。 也就是说，如果某个字段已经被图记录为可用状态字段，那么节点即使没有在返回类型注解中声明该字段，也可能仍然可以返回并更新它。 不过，为了代码清晰，仍然推荐让节点的返回值和返回类型注解保持一致。

5. 最终输出会按照 `output_schema` 进行裁剪 图运行完成后，最终返回给外部调用方的结果会按照 `output_schema` 进行裁剪。 因此，`output_schema` 的作用**不是“只让最后一个节点可见”，而是约束图最终对外暴露哪些字段**。 例如，图内部状态中可能同时存在：
   ```python
   username
   nickname
   greeting
   graph_output
   ```
    但如果 `output_schema` 只包含：

   ```python
   graph_output
   ```
   那么最终 `graph.invoke()` 的返回结果就只会包含 `graph_output`。

In [10]:
from langgraph.graph import START, END, StateGraph
from typing import TypedDict


class InputState(TypedDict):
    """输入状态"""
    username: str

class OutputState(TypedDict):
    """输出状态"""
    graph_output: str

class OverAllState(TypedDict):
    """全局状态"""
    username: str
    graph_output: str
    nickname: str

class PrivateState(TypedDict):
    """私有状态"""
    greeting: str


def node_1(state: InputState) -> OverAllState:
    """向全局状态添加nickname"""
    return {
        "nickname": "Dear " + state["username"]
    }

def node_2(state: OverAllState) -> PrivateState:
    """向私有状态添加greeting"""
    return {
        "greeting": "hello, " + state["nickname"]
    }

def node_3(state: PrivateState) -> OutputState:
    """向输出状态添加graph_output"""
    return {
        "graph_output": state["greeting"] + "，很高兴认识你！"
    }


builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

graph = builder.compile()
result = graph.invoke({"username": "小任"})
print(result)

{'graph_output': 'hello, Dear 小任，很高兴认识你！'}
